In [1]:
#!pip install pdfplumber

In [2]:
#!pip install ipywidgets

In [3]:
# Import libraries
import pdfplumber
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Adding resume to jUpyter
from IPython.display import display
from ipywidgets import FileUpload

# Resume Upload in pdf format
upload = FileUpload(accept='.pdf', multiple = True)
display(upload)

FileUpload(value=(), accept='.pdf', description='Upload', multiple=True)

In [4]:
all_resumes = []
if upload.value:
    # Accessing the uploaded file
    for uploaded_file in upload.value:
    
        # Save the PDF locally
        filename = uploaded_file['name']
        pdf_content = uploaded_file['content']
        with open(filename,"wb") as f:
            f.write(pdf_content)
        print(f"{filename} saved Successfully")
else:
    print("No files uploaded")

GEETHARANI Resume.pdf saved Successfully
Lekhya.pdf saved Successfully
Sangeetha.pdf saved Successfully
Shivani.pdf saved Successfully


In [5]:
for uploaded_file in upload.value:

    # Save PDF
    filename = uploaded_file['name']
    pdf_content = uploaded_file['content']

    with open(filename, "wb") as f:
        f.write(pdf_content)

    # Extract resume text
    resume_text = ""

    with pdfplumber.open(filename) as pdf:

        for page in pdf.pages:

            text = page.extract_text()

            if text:
                resume_text += text

    # Store data
    all_resumes.append({
        "filename": filename,
        "text": resume_text
    })

    # Print EACH resume
    print("=" * 50)
    print("Filename:", filename)
    print("=" * 50)

    print(resume_text[:2000])

    print("\n\n")

Filename: GEETHARANI Resume.pdf
GEETHARANI RAMACHANDRAN
Mobile : +91 99526 72312 | Email : rgeetharani94@gmail.com
LinkedIn: geetharani-ramachandran | GitHub: Geetharani-CodeAI
Professional Summary
Generative AI Engineer (Aspiring) with expertise in building AI-powered applications using
Large Language Models (LLMs), Machine Learning, and Deep Learning. Skilled in Prompt
Engineering, NLP pipelines, and end-to-end AI system development. Experienced in
computer vision and real-time ML solutions, with a strong foundation in Python, SQL, and
scalable model deployment. Focused on developing intelligent automation systems and
GenAI-driven applications for real-world problem solving.
CORE SKILLS
Generative AI & LLMs: Prompt Engineering, Retrieval-Augmented Generation (RAG),
LLM Applications, OpenAI API, LangChain (Learning), Hugging Face Transformers,
Semantic Search, AI Chatbot Development
Programming & Tools: Python, SQL, Pandas, NumPy, Scikit-learn, TensorFlow, Keras
Machine Learning & Dee

In [ ]:
#pip uninstall sentence-transformers transformers huggingface_hub httpx -y

In [ ]:
!pip install sentence-transformers==2.7.0
!pip install transformers==4.41.2
!pip install torch==2.3.1
!pip install faiss-cpu==1.8.0
!pip install streamlit==1.35.0

In [6]:
import faiss
import numpy as np

In [8]:
model = SentenceTransformer('all-MiniLM-L6-v2')

job_description = """
Looking for AI Engineer with Python, NLP and Machine Learning skills.
"""

# Convert text to Embeddings

jd_embedding = model.encode([job_description]).astype("float32")
faiss.normalize_L2(jd_embedding)

# Comparing all the resumes

resume_embeddings = []
for resume in all_resumes:
    embedding = model.encode(resume["text"])
    resume_embeddings.append(embedding)

# Convert to numpy float32
resume_embeddings = np.array(resume_embeddings).astype("float32")

# Normalize embedding
faiss.normalize_L2(resume_embeddings)

# Create FAISS index
dimension = resume_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# Add resume embedding
index.add(resume_embeddings)

# Searching Top 3 candidates
k = len(all_resumes)
scores, indexes = index.search(jd_embedding, k)


C:\snaptube\AnacondaPython\envs\genai_project\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


#### Ranking the resumes

In [9]:
# Create result list
results = []

for rank, idx in enumerate(indexes[0]):

    results.append({
        "filename": all_resumes[idx]["filename"],
        "score": scores[0][rank]
    })

# Sort candidates
ranked_results = sorted(
    results,
    key=lambda x: x["score"],
    reverse=True
)

# Display rankings
for rank, result in enumerate(ranked_results, start=1):

    print(f"Rank {rank}")

    print(f"Resume: {result['filename']}")

    print(f"Match Score: {round(result['score'] * 100, 2)}%")

    print("-" * 40)
    

Rank 1
Resume: GEETHARANI Resume.pdf
Match Score: 58.34%
----------------------------------------
Rank 2
Resume: Sangeetha.pdf
Match Score: 55.12%
----------------------------------------
Rank 3
Resume: Lekhya.pdf
Match Score: 45.52%
----------------------------------------


#### Creating Candidate Summary using LLM

In [14]:
#!pip install google-generativeai

In [10]:
import google.generativeai as genai
genai.configure(api_key = "API_Key")
model = genai.GenerativeModel("gemini-2.5-flash-lite")

top_candidates = ranked_results[:3]

for candidate in top_candidates:
    filename = candidate["filename"]
    score = candidate["score"]

    for resume in all_resumes:
        if resume["filename"] == filename:
            resume_text = resume["text"]
    prompt = f"""
You are an AI Hiring assistant.

Analyse this candidate against the job description.
Give output in this exact format:

1. Candidate Summary (2 Lines only)
2. Strengths  (3 points only)
3. Missing Skills (3 points only )
4. Hiring Recommendation (1 point only)

Resume: {resume_text}
Job Description: {job_description}
"""
    response = model.generate_content(prompt, generation_config = {"temperature": 0, "max_output_tokens" : 500})
    print(" "*30)
    print(f"Candidate:{filename}")
    print(" "*30)
    print(f"Match score:{round(score*100,2)}%")
    print(" "*30)
    print(response.text)
    print(""*30)


C:\snaptube\AnacondaPython\envs\genai_project\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\SSY MOBILES\AppData\Local\Temp\ipykernel_13356\998244344.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


                              
Candidate:GEETHARANI Resume.pdf
                              
Match score:58.34%
                              
1. Candidate Summary: Aspiring Generative AI Engineer with a strong foundation in Python, ML, and NLP, demonstrated through practical projects. Possesses experience in building AI applications, including resume screening and attrition prediction systems.
2. Strengths:
    * Proficient in Python, Pandas, NumPy, Scikit-learn, and core ML/DL concepts.
    * Hands-on experience with Generative AI tools like Gemini API, RAG, and Sentence Transformers.
    * Demonstrated ability to build end-to-end AI applications with deployment experience (Django, Streamlit).
3. Missing Skills:
    * Explicit mention of advanced NLP techniques beyond basic text processing.
    * Deeper experience with cloud deployment platforms (AWS, Azure, GCP).
    * Experience with MLOps practices for production-level AI systems.
4. Hiring Recommendation: Recommend for an interv

#### Relevant question retrieval based on the skillset mentioned in the resume using RAG

#### Creating Question Database

In [11]:
question_bank = {
    "Python": [
        "Explain list vs tuple.",
        "What are decorators in Python?",
        "Explain generators."
    ],

    "Machine Learning": [
        "What is overfitting?",
        "Explain bias vs variance.",
        "Difference between supervised and unsupervised learning?"
    ],
    "NLP": [
        "What is TF-IDF?",
        "Explain tokenization.",
        "What are embeddings?"
    ],

    "Deep Learning": [
        "Explain CNN architecture.",
        "What is backpropagation?",
        "Difference between CNN and RNN?"
    ],

    "SQL": [
        "Difference between WHERE and HAVING?",
        "Explain joins.",
        "What is normalization?"
    ]
}

 # Extract skills from Resume
skill_keywords = [
    "Python",
    "Machine Learning",
    "NLP",
    "Deep Learning",
    "SQL",
]
for candidate in top_candidates:
    filename = candidate["filename"]
    for resume in all_resumes:
        if resume["filename"] == filename:
            resume_text =resume["text"]
    skills = []
    for skill in skill_keywords:
            if skill.lower() in resume_text.lower():
                skills.append(skill)
    # Retrieve Questions
    retrieved_questions = []
    
    for skill in skills:
        if skill in question_bank:
            retrieved_questions.extend(question_bank[skill])
    # Display Output
    print("=" *30)
    print("Candidate:", filename)
    print("Skills:", skills)
    print("\n Basic Interview Questions:")
    print("=" *30)
    for q in retrieved_questions:
        print(q)
    print("=" *30)

    # Prompting 
    prompt = f"""
    You are an AI Technical Interviewer.
    
    Candidate Skills:
    {skills}
    
    Reference Interview Questions:
    {retrieved_questions}
    
    Generate:
    1. 5 technical interview questions
    2. 2 scenario-based questions
    3. 1 project-based question
    
    Keep questions personalized to the candidate profile
    """
    
    response = model.generate_content(prompt, generation_config = {"temperature": 0.5,
            "max_output_tokens": 400})
    
    print(response.text)

Candidate: GEETHARANI Resume.pdf
Skills: ['Python', 'Machine Learning', 'NLP', 'Deep Learning', 'SQL']

 Basic Interview Questions:
Explain list vs tuple.
What are decorators in Python?
Explain generators.
What is overfitting?
Explain bias vs variance.
Difference between supervised and unsupervised learning?
What is TF-IDF?
Explain tokenization.
What are embeddings?
Explain CNN architecture.
What is backpropagation?
Difference between CNN and RNN?
Difference between WHERE and HAVING?
Explain joins.
What is normalization?
Alright, let's get started. Based on your skills in Python, Machine Learning, NLP, Deep Learning, and SQL, I've prepared a set of questions designed to explore your expertise.

Here are your interview questions:

### Technical Interview Questions:

1.  **Python & Generality:** You've listed Python as a core skill. Could you explain the difference between a list and a tuple in Python, and in what scenarios would you choose one over the other?
2.  **Machine Learning Fund